In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, accuracy_score
import numpy as np
import itertools
import random
from tensorflow import keras
from tensorflow.keras import layers, regularizers, optimizers, callbacks
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, accuracy_score, f1_score

/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (
2026-05-22 15:31:04.192141: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [14]:
 # ── 1. Load dataset ───────────────────────────────────────────────────────────
file_path = "../1.DATASET/CMI_FINAL_OD.csv"
df=pd.read_csv(file_path)   # ← change to your file path

# ── 2. Separate features and target ──────────────────────────────────────────
X = df.drop(columns=['sii'])        # ← change 'target' to your label column
y = df['sii'].values

In [15]:
# ── 3. Split: Train 70% | Validation 15% | Test 15% ──────────────────────────
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y          # keeps class balance in each split
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,     # 50% of the 30% → 15% of total each
    random_state=42,
    stratify=y_temp
)
over_dic={1:2500,2:2000,3:1500}
#over_dic = {1:1500, 2:1000, 3:300}
over = SMOTE(random_state=42, sampling_strategy=over_dic)
X_train, y_train = over.fit_resample(X_train, y_train)

print(f"Train:      {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test:       {X_test.shape[0]} samples")

import numpy as np

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

Train:      10060 samples
Validation: 1259 samples
Test:       1260 samples


/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [16]:
def build_nn(hidden_layers, activation, reg_type, reg_penalty, lr, momentum, optimizer_name, n_classes):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(X_train_scaled.shape[1],)))
    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation,
                               kernel_regularizer=reg_type(reg_penalty)))
    model.add(layers.Dense(n_classes, activation='softmax',
                           kernel_regularizer=reg_type(reg_penalty)))

    if optimizer_name == 'sgd':   # ← now properly refers to the parameter
        opt = optimizers.SGD(learning_rate=lr, momentum=momentum)
    else:
        opt = optimizers.Adam(learning_rate=lr)

    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=opt, metrics=['accuracy'])
    return model

In [17]:
# ── 2. Parameter grid ─────────────────────────────────────────────────────────
param_grid = {
    'hidden_layers': [[4, 8], [32, 64, 32], [64, 128, 64], [64, 32], [128, 64]],
    'activation':    ['relu'],
    'optimizer':     ['adam', 'sgd'],
    'lr':            [0.01, 0.005, 0.001, 1e-3, 5e-4, 1e-4],
    'reg_type':      [regularizers.l2],
    'reg_penalty':   [0.005, 0.001, 1e-4, 1e-5],
    'momentum':      [0.9, 0.5, 0.1],   # ignored when optimizer=adam
    'epochs':        [100, 500, 1000],
    'batch_size':    [32, 64, 128],
}

In [18]:
# ── 3. Random search with k-fold CV ──────────────────────────────────────────
N_ITER   = 30     # number of random combos to try
N_FOLDS  = 3      # k for cross-validation
RANDOM_STATE = 42
N_CLASSES = len(np.unique(y_train))  # → 4 in your case

random.seed(RANDOM_STATE)

# Sample N_ITER random combinations
keys   = list(param_grid.keys())
combos = []
seen   = set()
while len(combos) < N_ITER:
    combo = {k: random.choice(param_grid[k]) for k in keys}
    key   = str(combo)
    if key not in seen:
        seen.add(key)
        combos.append(combo)

kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

best_score  = -np.inf
best_params = None
results     = []

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight = dict(enumerate(class_weights_array))

for i, params in enumerate(combos):
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_scaled, y_train)):
        X_f_train, X_f_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_f_train, y_f_val = y_train[train_idx],        y_train[val_idx]

        #X_f_train, y_f_train = over.fit_resample(X_f_train, y_f_train)

        # Use adam lr if optimizer is adam, else sgd lr
        lr = params['lr'] if params['optimizer'] == 'sgd' else params['lr']

        model = build_nn(
            hidden_layers=params['hidden_layers'],
            activation=params['activation'],
            reg_type=params['reg_type'],
            reg_penalty=params['reg_penalty'],
            lr=lr,
            momentum=params['momentum'] if params['optimizer'] == 'sgd' else 0.0,
            optimizer_name=params['optimizer'],   # ← add this
            n_classes=N_CLASSES
        )


        # Light early stopping during search to save time
        es = callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                     min_delta=1e-4, verbose=0)

        model.fit(X_f_train, y_f_train,
                  epochs=params['epochs'],
                  batch_size=params['batch_size'],
                  validation_data=(X_f_val, y_f_val),
                  #class_weight=class_weight,
                  callbacks=[es],
                  verbose=0)

        _, acc = model.evaluate(X_f_val, y_f_val, verbose=0)
        fold_scores.append(acc)

        """y_f_pred = np.argmax(model.predict(X_f_val, verbose=0), axis=1)
        score = f1_score(y_f_val, y_f_pred, average='macro', zero_division=0)
        fold_scores.append(score)"""

    mean_score = np.mean(fold_scores)
    results.append({'params': params, 'score': mean_score})
    print(f"[{i+1:2d}/{N_ITER}] score={mean_score:.4f} | {params}")

    if mean_score > best_score:
        best_score  = mean_score
        best_params = params

print("\nBest CV score:", best_score)
print("Best params:  ", best_params) 

[ 1/30] score=0.4374 | {'hidden_layers': [4, 8], 'activation': 'relu', 'optimizer': 'sgd', 'lr': 0.005, 'reg_type': <class 'keras.src.regularizers.regularizers.L2'>, 'reg_penalty': 0.001, 'momentum': 0.1, 'epochs': 100, 'batch_size': 128}
[ 2/30] score=0.5748 | {'hidden_layers': [128, 64], 'activation': 'relu', 'optimizer': 'sgd', 'lr': 0.01, 'reg_type': <class 'keras.src.regularizers.regularizers.L2'>, 'reg_penalty': 0.005, 'momentum': 0.9, 'epochs': 100, 'batch_size': 128}
[ 3/30] score=0.6230 | {'hidden_layers': [128, 64], 'activation': 'relu', 'optimizer': 'adam', 'lr': 0.0001, 'reg_type': <class 'keras.src.regularizers.regularizers.L2'>, 'reg_penalty': 0.001, 'momentum': 0.5, 'epochs': 1000, 'batch_size': 64}
[ 4/30] score=0.4980 | {'hidden_layers': [4, 8], 'activation': 'relu', 'optimizer': 'sgd', 'lr': 0.001, 'reg_type': <class 'keras.src.regularizers.regularizers.L2'>, 'reg_penalty': 0.001, 'momentum': 0.9, 'epochs': 500, 'batch_size': 32}
[ 5/30] score=0.5005 | {'hidden_layers

In [19]:
# ── 4. Final retrain with best params + proper early stopping ─────────────────
if best_params['optimizer'] == 'adam':
    opt = optimizers.Adam(learning_rate=best_params['lr'])
else:
    opt = optimizers.SGD(learning_rate=best_params['lr'],
                         momentum=best_params['momentum'])

final_model = build_nn(
    hidden_layers=best_params['hidden_layers'],
    activation=best_params['activation'],
    reg_type=best_params['reg_type'],
    reg_penalty=best_params['reg_penalty'],
    lr=best_params['lr'],
    momentum=best_params['momentum'] if best_params['optimizer'] == 'sgd' else 0.0,
    optimizer_name=best_params['optimizer'],  # ← add this
    n_classes=N_CLASSES,  
)

es = callbacks.EarlyStopping(monitor='val_loss', patience=10,
                              min_delta=1e-4, verbose=True)
mc = callbacks.ModelCheckpoint('best_model.keras', monitor='val_loss',
                                save_best_only=True)

final_model.fit(
    X_train_scaled, y_train,
    epochs=1000,
    batch_size=best_params['batch_size'],
    shuffle=True,
    #class_weight=class_weight,
    validation_data=(X_val_scaled, y_val),
    callbacks=[es, mc]) 

Epoch 1/1000
315/315 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4480 - loss: 1.2229 - val_accuracy: 0.5925 - val_loss: 1.0265
Epoch 2/1000
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5365 - loss: 1.0663 - val_accuracy: 0.5743 - val_loss: 1.0450
Epoch 3/1000
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5756 - loss: 0.9892 - val_accuracy: 0.5488 - val_loss: 1.0568
Epoch 4/1000
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5945 - loss: 0.9362 - val_accuracy: 0.5981 - val_loss: 0.9842
Epoch 5/1000
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6172 - loss: 0.8966 - val_accuracy: 0.6156 - val_loss: 0.9917
Epoch 6/1000
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6310 - loss: 0.8624 - val_accuracy: 0.5735 - val_loss: 1.0354
Epoch 7/1000
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6468 - loss: 0.8372 - val_accuracy: 0.5647 - val_loss: 1.0463
Epoch 8/1000
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6496 - loss: 0.8172 - 

In [20]:
# ── 5. Final evaluation on test set ──────────────────────────────────────────

best_model_loaded = keras.models.load_model('best_model.keras')
y_pred = np.argmax(best_model_loaded.predict(X_test_scaled), axis=1)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('F1 macro:', f1_score(y_test, y_pred, average='macro'))
print('F1 per class:', f1_score(y_test, y_pred, average=None))
print(classification_report(y_test, y_pred))

40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Accuracy: 0.5992063492063492
F1 macro: 0.29766477809949726
F1 per class: [0.75645342 0.25820569 0.176      0.        ]
              precision    recall  f1-score   support

         0.0       0.74      0.77      0.76       871
         1.0       0.27      0.25      0.26       236
         2.0       0.20      0.16      0.18       141
         3.0       0.00      0.00      0.00        12

    accuracy                           0.60      1260
   macro avg       0.30      0.29      0.30      1260
weighted avg       0.58      0.60      0.59      1260

